In [1]:
from pathlib import Path
import pandas as pd
from pyreaddbc import dbc2dbf
from dbfread import DBF

Vamos definir os caminhos das pastas utilizadas no processo de conversão.

A pasta `dados/raw` vai conter os arquivos brutos originais em formato DBC, enquanto a pasta `dados/interim` irá armazenar os arquivos intermediários gerados após a conversão.

Caso a pasta não exista, ela será criada.

In [2]:
pasta_raw = Path("dados/raw")
pasta_interim = Path("dados/interim")

pasta_interim.mkdir(parents=True, exist_ok=True)

Com isso feito, os arquivos DBC presentes na pasta `dados/raw` serão convertidos para o formato DBF, lidos como DataFrames e, então, cada arquivo será salvo individualmente em formato CSV na pasta `dados/interim`, facilitando a manipulação posterior, tendo em vista que o formato CSV é mais simples de carregar e analisar.

In [3]:
for arquivo_dbc in pasta_raw.glob("*.dbc"):
    print(f"Convertendo {arquivo_dbc.name}...")

    arquivo_dbf = pasta_interim / f"{arquivo_dbc.stem}.dbf"
    arquivo_csv = pasta_interim / f"{arquivo_dbc.stem}.csv"

    dbc2dbf(str(arquivo_dbc), str(arquivo_dbf))

    tabela = DBF(
        str(arquivo_dbf),
        encoding="latin1",
        char_decode_errors="ignore"
    )

    df_temp = pd.DataFrame(iter(tabela))

    df_temp.to_csv(
        arquivo_csv,
        index=False,
        sep=";",
        encoding="utf-8-sig"
    )

    print(f"Salvo: {arquivo_csv.name}")

arquivos_csv = list(Path("dados/interim").glob("RD*.csv"))


Convertendo RDMT2001.dbc...
Salvo: RDMT2001.csv
Convertendo RDMT2002.dbc...
Salvo: RDMT2002.csv
Convertendo RDMT2003.dbc...
Salvo: RDMT2003.csv
Convertendo RDMT2004.dbc...
Salvo: RDMT2004.csv
Convertendo RDMT2005.dbc...
Salvo: RDMT2005.csv
Convertendo RDMT2006.dbc...
Salvo: RDMT2006.csv
Convertendo RDMT2007.dbc...
Salvo: RDMT2007.csv
Convertendo RDMT2008.dbc...
Salvo: RDMT2008.csv
Convertendo RDMT2009.dbc...
Salvo: RDMT2009.csv
Convertendo RDMT2010.dbc...
Salvo: RDMT2010.csv
Convertendo RDMT2011.dbc...
Salvo: RDMT2011.csv
Convertendo RDMT2012.dbc...
Salvo: RDMT2012.csv
Convertendo RDMT2101.dbc...
Salvo: RDMT2101.csv
Convertendo RDMT2102.dbc...
Salvo: RDMT2102.csv
Convertendo RDMT2103.dbc...
Salvo: RDMT2103.csv
Convertendo RDMT2104.dbc...
Salvo: RDMT2104.csv
Convertendo RDMT2105.dbc...
Salvo: RDMT2105.csv
Convertendo RDMT2106.dbc...
Salvo: RDMT2106.csv
Convertendo RDMT2107.dbc...
Salvo: RDMT2107.csv
Convertendo RDMT2108.dbc...
Salvo: RDMT2108.csv
Convertendo RDMT2109.dbc...
Salvo: RDMT2

Após a conversão, os arquivos CSV serão carregados e unidos em um único DataFrame. Também será criada uma coluna indicando a origem do arquivo, permitindo-me identificar qual registro veio de qual arquivo, o que é útil para conferência, rastreabilidade e validação dos dados.

In [4]:
dfs = []

for arquivo in arquivos_csv:
    df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
    df_temp["arquivo_origem"] = arquivo.name
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

df.shape

pasta_processed = Path("dados/processed")
pasta_processed.mkdir(parents=True, exist_ok=True)

C:\Users\djmet\AppData\Local\Temp\ipykernel_38756\549544787.py:4: DtypeWarning: Columns (0: DIAGSEC2, 1: DIAGSEC3, 2: DIAGSEC4, 3: DIAGSEC5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
C:\Users\djmet\AppData\Local\Temp\ipykernel_38756\549544787.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_temp["arquivo_origem"] = arquivo.name
C:\Users\djmet\AppData\Local\Temp\ipykernel_38756\549544787.py:4: DtypeWarning: Columns (0: DIAGSEC2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8-sig")
C:\Users\djmet\AppData\Local\Temp\ipykernel_38756\549544787.py:5: PerformanceWarning: DataFrame

Com isso feito, vou salvar a base consolidada. Essa será a base principal, utilizada nas próximas etapas do projeto.

In [5]:
caminho_base = Path("dados/processed/sih_sus_mt_internacoes_completo.csv")

df = pd.read_csv(
    caminho_base,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

print(f"Arquivo salvo em: {caminho_base}")

Arquivo salvo em: dados\processed\sih_sus_mt_internacoes_completo.csv
